# NB153: Wave Propagation on the Solenoid

**The question**: The mass exponent x = 3.000376 for leptons (not exactly 3). What produces this specific value? Not character counting, not covering level counting — the actual wave propagation physics on the solenoid structure.

**The approach**: Treat the cascade as waves propagating through four concentric covering maps. At each level, the wave:
- Enters from the inner orbit (driven by sin(θ_{k-1}))  
- Is attenuated by the covering map (coupling 1/p_k)
- Is damped by the containment (rate κ)
- Wraps around the circle if amplitude exceeds π

The exponent should emerge as the TOTAL PROPAGATION GAIN from the base to the measurement level. Each covering map contributes a gain factor. The product of all gain factors IS the exponent.

The 0.000376 correction over the integer 3 should be traceable to the specific wave coupling at each level — the interference between the driven response and the natural frequency at each covering map.

## S0: The Solenoid as a Waveguide

The covering tower S¹ ←2← S¹ ←3← S¹ ←5← S¹ ←7← S¹ is a chain of concentric waveguides. Each level is a circle. Each covering map connects one circle to the next through a p-fold wrapping.

A wave on the base circle (frequency ω = 2π) propagates upward through the tower. At each level k, the wave:

1. **Arrives** with amplitude A_k from level k-1, frequency ω/P_{k-1}
2. **Couples** through the covering map: the p_k-fold cover maps one wave period to p_k periods on the next circle
3. **Resonates** or **attenuates** depending on how the wave frequency relates to the natural frequency at level k
4. **Departs** with amplitude A_{k+1} toward level k+1

The mass exponent is the TOTAL logarithmic gain: x = Σ_k log(A_{k+1}/A_k) / log(CP_base)

Each level contributes a gain factor. The product of gains across all levels IS the exponent.

Let me compute the per-level gain from the actual cascade data.

In [3]:
# ── S0: Per-level wave propagation gain ──

import sys, os, time, numpy as np
from pathlib import Path
from math import gcd, prod

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA, KAPPA, EPSILON, OMEGA
from solenoid_system import SolenoidSystem
from solenoid_jax import warmup as jax_warmup, detect_device

print('S0: THE SOLENOID AS A WAVEGUIDE — PER-LEVEL PROPAGATION')
print('='*70)

primes = SA.primes
P4 = SA.P
kappa = KAPPA
omega = OMEGA

# Integrate the cascade
sys0 = SolenoidSystem()
all_branches = sys0.all_branches()
cis = SA.coprime_indices(P4)

jax_warmup()
t_eval = cis.astype(float)
res = sys0.integrate_all_branches(all_branches, t_eval, float(P4+1), backend='jax')

# Compute sector RMS at all levels
rms = np.zeros((len(cis), 4))
for idx in range(len(cis)):
    for k in range(4):
        Rk = np.array([res[br][idx, k] for br in all_branches])
        Rk_w = np.mod(Rk, 2*np.pi)
        Rk_w[Rk_w > np.pi] -= 2*np.pi
        rms[idx, k] = np.sqrt(np.mean(Rk_w**2))

# The LEPTON CP pair: ci=31 (g1) and ci=61 (g2)
idx_g1 = np.where(cis == 31)[0][0]
idx_g2 = np.where(cis == 61)[0][0]

# CP ratio at each level
print(f'LEPTON pair (ci=31/61) — CP ratio and per-level gain:')
print(f'  {"Level":>6s}  {"CP_k":>10s}  {"ln(CP_k)":>10s}  {"Δln(CP)":>10s}  {"gain_k":>10s}')

cp_per_level = []
ln_cp = []
for k in range(4):
    cp_k = rms[idx_g1, k] / rms[idx_g2, k]
    cp_per_level.append(cp_k)
    ln_cp.append(np.log(cp_k))

# The per-level gain: how much does the CP ratio CHANGE from level k to k+1?
# If the wave propagates through covering map k+1, the CP should change.
# gain_k = ln(CP_{k+1}) / ln(CP_k)
# This is the cross-level factor from NB137.
for k in range(4):
    if k > 0:
        delta = ln_cp[k] - ln_cp[k-1]
        gain = ln_cp[k] / ln_cp[k-1]
    else:
        delta = ln_cp[0]
        gain = 0  # no previous level
    print(f'  R{k:1d}      {cp_per_level[k]:10.4f}  {ln_cp[k]:10.6f}  '
          f'{delta:10.6f}  {gain:10.6f}') if k > 0 else print(f'  R{k:1d}      {cp_per_level[k]:10.4f}  {ln_cp[k]:10.6f}  {"—":>10s}  {"—":>10s}')

# The TOTAL exponent at R3:
x_R3 = np.log(206.768) / ln_cp[3]  # using PDG m_mu/m_e
x_R3_actual = np.log(206.768) / np.log(cp_per_level[3])
print(f'\n  Total exponent at R3: x = ln(206.768)/ln(CP_R3) = {x_R3_actual:.6f}')
print(f'  Deviation from 3: δ = {x_R3_actual - 3:.6f}')

# Per-level exponent contribution:
# If x = product of per-level gains, then log(x) = sum of per-level log-gains
# x(R3) / x(R0) = cross-level = ln(CP_R0)/ln(CP_R3)
cross_level = ln_cp[0] / ln_cp[3]
x_R0 = x_R3_actual / cross_level
print(f'  Cross-level R0→R3: {cross_level:.6f}')
print(f'  x(R0) = {x_R0:.6f}')
print(f'  x(R0) × cross-level = {x_R0 * cross_level:.6f} ← should be x(R3)')

# Now: the per-level contribution to the cross-level
print(f'\n  Per-level cross-level decomposition:')
for k in range(1, 4):
    cl_step = ln_cp[k-1] / ln_cp[k]  # how much does level k change the exponent?
    print(f'    R{k-1}→R{k}: ln(CP_{k-1})/ln(CP_{k}) = {cl_step:.6f} '
          f'(prime p{k+1}={primes[k]})')

# The cross-level is the PRODUCT of per-step factors:
product_cl = 1.0
for k in range(1, 4):
    cl_step = ln_cp[k-1] / ln_cp[k]
    product_cl *= cl_step
print(f'  Product of per-step cross-levels: {product_cl:.6f}')
print(f'  Total cross-level: {cross_level:.6f}')
print(f'  Match: {abs(product_cl - cross_level) < 1e-10}')

# NOW: what determines each per-step cross-level?
# At level k, the wave enters with CP from level k-1 and exits with CP at level k.
# The change is due to the coupling through the covering map of degree p_{k+1}.
# The CASCADE ODE at level k+1 is:
# dR_{k+1}/dt + κR_{k+1} = ε sin(θ_{k+1}) - ε sin(θ_k)/p_{k+1} + κ R_k/p_{k+1}
# 
# The LINEAR part (ignoring wrapping) gives a filter with gain:
# H_k = 1/sqrt(1 + (ωP_k/(κP_{k+1}))²)
# Wait — that's from NB107. Let me use the actual filter gains.

primorials = [1, 2, 6, 30, 210]
print(f'\n\nPER-LEVEL FILTER ANALYSIS:')
print(f'  The cascade filter gain at each level (NB107):')
for k in range(4):
    Pk = primorials[k]
    H_sq = Pk**2 / (Pk**2 + omega**2 * P4)
    H = np.sqrt(H_sq)
    print(f'  R{k}: |H|² = {Pk}²/({Pk}² + ω²·{P4}) = {H_sq:.6f}, |H| = {H:.6f}')

# The CP ratio at each level is related to the filter gain:
# CP_k ∝ 1/|H_k| × CP_{k-1}^{something}
# Actually no — the CP ratio at each level is determined by the WRAPPING
# at that level, not by the filter gain.

# Let me think about this differently.
# The CP ratio measures the CONTRAST between g1 and g2 at each level.
# At R0: CP_R0 = RMS(g1)/RMS(g2). This is set by the transient decay.
# At R1: the R0 signal propagates through the p1=2 covering map.
# At R2: the R1 signal propagates through the p2=3 covering map.
# At R3: the R2 signal propagates through the p3=5 covering map.

# The PROPAGATION changes the CP ratio because:
# - The signal (transient) decays at the same rate κ at all levels
# - But the COUPLING is 1/p_k at each level
# - And the WRAPPING threshold is the same (±π) at all levels
# - So the EFFECTIVE attenuation at each level depends on p_k

# The per-step cross-level is: how much does the CP ratio change
# when the signal passes through one covering map?

print(f'\n  Per-step cross-levels and their relationship to primes:')
for k in range(1, 4):
    cl_step = ln_cp[k-1] / ln_cp[k]
    pk = primes[k]  # the covering map at this step
    print(f'    R{k-1}→R{k} (p={pk}): cross-level = {cl_step:.6f}')
    print(f'      1/p = {1/pk:.6f}, (p-1)/p = {(pk-1)/pk:.6f}, p/(p+1) = {pk/(pk+1):.6f}')
    print(f'      cl - 1 = {cl_step - 1:.6f}')

# The cross-level at each step should be close to 1 + correction.
# The correction depends on how the p-fold covering map transforms the signal.

# At R0→R1 (p=2): the signal splits into 2 sheets.
#   Each sheet sees half the driving force.
#   But the R1 also has its own transient (j2) adding independent signal.
# At R1→R2 (p=3): splits into 3 sheets.
# At R2→R3 (p=5): splits into 5 sheets.

# The CP ratio CHANGES because the wrapping pattern changes.
# At the g1 crossing (inside wrapping zone):
#   More sheets → more branches wrapped → different compression
# At the g2 crossing (outside wrapping zone):
#   More sheets → but all in linear regime → no change to CP

# Since g2 is exactly linear (NB149), the CP at each level is:
# CP_k = RMS_wrapped(g1, level k) / RMS_linear(g2, level k)
# And RMS_linear scales predictably with the filter gain.
# The cross-level is driven by how WRAPPING changes from level to level.

print(f'\n  WRAPPING FRACTION at g1 (ci=31) per level:')
for k in range(4):
    Rk_g1 = np.array([res[br][idx_g1, k] for br in all_branches])
    n_wrap = np.sum(np.abs(Rk_g1) > np.pi)
    pct = 100 * n_wrap / len(all_branches)
    print(f'    R{k}: {n_wrap}/{len(all_branches)} = {pct:.1f}% wrapped')

print(f'\n  WRAPPING FRACTION at g2 (ci=61) per level:')
for k in range(4):
    Rk_g2 = np.array([res[br][idx_g2, k] for br in all_branches])
    n_wrap = np.sum(np.abs(Rk_g2) > np.pi)
    pct = 100 * n_wrap / len(all_branches)
    print(f'    R{k}: {n_wrap}/{len(all_branches)} = {pct:.1f}% wrapped')


S0: THE SOLENOID AS A WAVEGUIDE — PER-LEVEL PROPAGATION
  JAX [CPU (1 device(s))]: 210 branches, 48 eval pts, T=211.0 — 2.56s
LEPTON pair (ci=31/61) — CP ratio and per-level gain:
   Level        CP_k    ln(CP_k)     Δln(CP)      gain_k
  R0          8.7738    2.171772           —           —
  R1          5.4299    1.691919   -0.479853    0.779050
  R2          5.2273    1.653894   -0.038025    0.977525
  R3          5.9120    1.776977    0.123083    1.074420

  Total exponent at R3: x = ln(206.768)/ln(CP_R3) = 3.000376
  Deviation from 3: δ = 0.000376
  Cross-level R0→R3: 1.222173
  x(R0) = 2.454953
  x(R0) × cross-level = 3.000376 ← should be x(R3)

  Per-level cross-level decomposition:
    R0→R1: ln(CP_0)/ln(CP_1) = 1.283615 (prime p2=3)
    R1→R2: ln(CP_1)/ln(CP_2) = 1.022991 (prime p3=5)
    R2→R3: ln(CP_2)/ln(CP_3) = 0.930735 (prime p4=7)
  Product of per-step cross-levels: 1.222173
  Total cross-level: 1.222173
  Match: True


PER-LEVEL FILTER ANALYSIS:
  The cascade filter ga